In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root / "src"))

# Loss function experiments

Demonstrate the TinyTorch loss functions `MSELoss_CP` and `CrossEntropy` and compare them against their PyTorch equivalents. Each loss is checked for correctness first, then explored on a small synthetic dataset, and finally timed against PyTorch.

- `MSELoss_CP.forward` computes `mean((prediction - target)**2)` and returns a scalar `Tensor_CP`. For 1-D targets this equals `torch.nn.MSELoss` with the default `reduction="mean"`.
- `CrossEntropy.forward` applies a numerically stable `log_softmax` (log-sum-exp trick) and returns the mean negative log-likelihood, matching `torch.nn.CrossEntropyLoss` on raw logits with integer class targets.

In [3]:
from statistics import median
from time import perf_counter

import numpy as np
import torch
from torch import nn

from tinytorch.loss_functions import CrossEntropy, MSELoss_CP
from tinytorch.tensor import Tensor_CP

np.random.seed(42)
torch.manual_seed(42)

print(f"NumPy:   {np.__version__}")
print(f"PyTorch: {torch.__version__}")
print("Device:  CPU")

NumPy:   2.5.0
PyTorch: 2.12.1+cpu
Device:  CPU


## Mean squared error

### Synthetic dataset

We generate a simple linear relationship `y = true_slope * x + true_bias` and add Gaussian noise so the targets are not perfectly linear. This mirrors a real regression problem where the model can never drive the loss to exactly zero.

In [4]:
rng = np.random.default_rng(7)

sample_count = 256
true_slope = 2.0
true_bias = -1.0
noise_std = 0.5

x = rng.uniform(-3.0, 3.0, size=sample_count).astype(np.float32)
noise = rng.normal(0.0, noise_std, size=sample_count).astype(np.float32)
y = (true_slope * x + true_bias + noise).astype(np.float32)

print(f"Samples: {sample_count}")
print(f"True relationship: y = {true_slope} * x + ({true_bias}) + noise(std={noise_std})")
print(f"x range: [{x.min():.2f}, {x.max():.2f}]")
print(f"y range: [{y.min():.2f}, {y.max():.2f}]")

Samples: 256
True relationship: y = 2.0 * x + (-1.0) + noise(std=0.5)
x range: [-2.98, 2.97]
y range: [-7.91, 5.06]


### Correctness against PyTorch

We use the noisy targets from the true parameters as the model prediction, so the only error left is the noise. `MSELoss_CP` should report the same loss as `torch.nn.MSELoss`.

In [5]:
def mse_cp(predictions, targets):
    """Mean squared error from MSELoss_CP (returns a scalar Tensor_CP)."""
    loss = MSELoss_CP().forward(Tensor_CP(predictions), Tensor_CP(targets))
    return float(loss.data)


predictions = (true_slope * x + true_bias).astype(np.float32)

loss_cp = mse_cp(predictions, y)
loss_torch = float(
    nn.MSELoss()(torch.from_numpy(predictions), torch.from_numpy(y))
)

print(f"TinyTorch MSELoss_CP: {loss_cp:.6f}")
print(f"PyTorch  nn.MSELoss: {loss_torch:.6f}")
print(f"Absolute difference:  {abs(loss_cp - loss_torch):.2e}")

assert np.isclose(loss_cp, loss_torch, atol=1e-5), "MSE loss mismatch!"
print("MSELoss_CP matches PyTorch")

TinyTorch MSELoss_CP: 0.231074
PyTorch  nn.MSELoss: 0.231074
Absolute difference:  1.49e-08
MSELoss_CP matches PyTorch


### The loss reflects prediction quality

A model whose predictions are close to the targets should produce a smaller loss than a poor one. We compare the true-parameter model against a deliberately wrong model and against the trivial baseline of always predicting the mean of `y`.

In [6]:
good_predictions = (true_slope * x + true_bias).astype(np.float32)
bad_predictions = (-1.0 * x + 4.0).astype(np.float32)
mean_baseline = np.full_like(y, y.mean())

loss_good = mse_cp(good_predictions, y)
loss_bad = mse_cp(bad_predictions, y)
loss_baseline = mse_cp(mean_baseline, y)

print(f"Good model (true params):       {loss_good:.4f}")
print(f"Mean baseline (predict y.mean): {loss_baseline:.4f}")
print(f"Bad model (wrong slope):        {loss_bad:.4f}")

assert loss_good < loss_baseline < loss_bad
print("Loss ranks the models as expected: good < baseline < bad")

Good model (true params):       0.2311
Mean baseline (predict y.mean): 12.1080
Bad model (wrong slope):        51.9029
Loss ranks the models as expected: good < baseline < bad


### Loss landscape over the slope

Holding the bias at its true value, we sweep candidate slopes and record the loss for each. The loss forms a convex bowl whose minimum sits at the slope that generated the data, which is exactly why minimizing MSE recovers the underlying parameters.

In [7]:
candidate_slopes = np.linspace(true_slope - 2.0, true_slope + 2.0, 9)

print(f"{'slope':>8} | {'MSE loss':>10}")
print("-" * 22)
losses = []
for slope in candidate_slopes:
    candidate_predictions = (slope * x + true_bias).astype(np.float32)
    loss = mse_cp(candidate_predictions, y)
    losses.append(loss)
    marker = "  <- true slope" if np.isclose(slope, true_slope) else ""
    print(f"{slope:>8.2f} | {loss:>10.4f}{marker}")

best_slope = candidate_slopes[int(np.argmin(losses))]
print(f"\nLowest loss at slope = {best_slope:.2f} (true slope = {true_slope})")

assert np.isclose(best_slope, true_slope), "Loss minimum should be at the true slope!"
print("MSE is minimized at the true slope")

   slope |   MSE loss
----------------------
    0.00 |    12.1081
    0.50 |     6.8894
    1.00 |     3.1703
    1.50 |     0.9509
    2.00 |     0.2311  <- true slope
    2.50 |     1.0109
    3.00 |     3.2904
    3.50 |     7.0695
    4.00 |    12.3483

Lowest loss at slope = 2.00 (true slope = 2.0)
MSE is minimized at the true slope


## Cross-entropy loss

### Synthetic classification dataset

We create random logits over `class_count` classes together with random integer class targets. Cross-entropy takes raw logits (not probabilities) and the correct class index for each sample, exactly like `torch.nn.CrossEntropyLoss`.

In [8]:
class_count = 4
ce_sample_count = 256
ce_rng = np.random.default_rng(11)

class_targets = ce_rng.integers(0, class_count, size=ce_sample_count).astype(np.int64)
base_logits = ce_rng.normal(0.0, 1.0, size=(ce_sample_count, class_count)).astype(np.float32)

print(f"Samples: {ce_sample_count}, classes: {class_count}")
print(f"Logits shape: {base_logits.shape}")
print(f"First 5 targets: {class_targets[:5]}")

Samples: 256, classes: 4
Logits shape: (256, 4)
First 5 targets: [0 0 3 1 2]


### Correctness against PyTorch

`CrossEntropy.log_softmax` should match `torch.log_softmax`, and the full loss should match `torch.nn.CrossEntropyLoss` on the same logits and targets.

In [9]:
cross_entropy = CrossEntropy()

log_probs_cp = cross_entropy.log_softmax(Tensor_CP(base_logits)).data
log_probs_torch = torch.log_softmax(torch.from_numpy(base_logits), dim=-1).numpy()
log_softmax_diff = np.max(np.abs(log_probs_cp - log_probs_torch))

ce_cp = float(cross_entropy.forward(Tensor_CP(base_logits), Tensor_CP(class_targets)).data)
ce_torch = float(
    nn.CrossEntropyLoss()(torch.from_numpy(base_logits), torch.from_numpy(class_targets))
)

print(f"log_softmax max difference:   {log_softmax_diff:.2e}")
print(f"TinyTorch CrossEntropy:       {ce_cp:.6f}")
print(f"PyTorch  nn.CrossEntropyLoss: {ce_torch:.6f}")
print(f"Absolute difference:          {abs(ce_cp - ce_torch):.2e}")

assert np.allclose(log_probs_cp, log_probs_torch, atol=1e-5), "log_softmax mismatch!"
assert np.isclose(ce_cp, ce_torch, atol=1e-5), "CrossEntropy loss mismatch!"
print("CrossEntropy and log_softmax match PyTorch")

log_softmax max difference:   2.38e-07
TinyTorch CrossEntropy:       1.624034
PyTorch  nn.CrossEntropyLoss: 1.624034
Absolute difference:          1.19e-07
CrossEntropy and log_softmax match PyTorch


### Numerical stability (log-sum-exp trick)

`log_softmax` subtracts the row maximum before exponentiating, so very large logits do not overflow. A naive `exp(logits)` blows up to `inf`, but the loss stays finite.

In [10]:
huge_logits = np.array([[1000.0, 1001.0, 999.0]], dtype=np.float32)
huge_target = np.array([1], dtype=np.int64)

stable_loss = float(cross_entropy.forward(Tensor_CP(huge_logits), Tensor_CP(huge_target)).data)
naive_exp = np.exp(huge_logits.astype(np.float64))

print(f"Loss with logits near 1000: {stable_loss:.6f}")
print(f"Naive exp(logits):          {naive_exp}")

assert np.isfinite(stable_loss), "Loss should stay finite for large logits!"
print("log-sum-exp keeps the loss finite while naive exp overflows")

Loss with logits near 1000: 0.407606
Naive exp(logits):          [[inf inf inf]]
log-sum-exp keeps the loss finite while naive exp overflows


C:\Users\CBABM\AppData\Local\Temp\ipykernel_23940\2011696062.py:5: RuntimeWarning: overflow encountered in exp
  naive_exp = np.exp(huge_logits.astype(np.float64))


### Cross-entropy rewards confident, correct predictions

Starting from the random logits, we add a growing margin to each sample's true-class logit. As the margin increases the model becomes more confident about the correct class, so accuracy rises and the loss falls toward zero.

In [11]:
margins = np.linspace(0.0, 6.0, 7)

print(f"{'margin':>8} | {'CE loss':>10} | {'accuracy':>9}")
print("-" * 34)
ce_losses = []
for margin in margins:
    boosted_logits = base_logits.copy()
    boosted_logits[np.arange(ce_sample_count), class_targets] += margin
    loss = float(cross_entropy.forward(Tensor_CP(boosted_logits), Tensor_CP(class_targets)).data)
    accuracy = float(np.mean(boosted_logits.argmax(axis=1) == class_targets))
    ce_losses.append(loss)
    print(f"{margin:>8.1f} | {loss:>10.4f} | {accuracy:>9.3f}")

assert all(later < earlier for earlier, later in zip(ce_losses, ce_losses[1:])), \
    "Loss should fall as the true-class logit grows!"
print("\nLoss decreases as confidence in the correct class grows")

  margin |    CE loss |  accuracy
----------------------------------
     0.0 |     1.6240 |     0.270
     1.0 |     0.9819 |     0.605
     2.0 |     0.5285 |     0.848
     3.0 |     0.2539 |     0.977
     4.0 |     0.1109 |     0.992
     5.0 |     0.0452 |     1.000
     6.0 |     0.0176 |     1.000

Loss decreases as confidence in the correct class grows


## Efficiency

The timer warms up each loss and reports the median time per call. Tensors are created outside the timed region so the measurement focuses on `forward()`.

In [12]:
def benchmark(function, *, warmup=3, repeats=7, number=50):
    for _ in range(warmup):
        function()

    samples_ms = []
    for _ in range(repeats):
        start = perf_counter()
        for _ in range(number):
            function()
        samples_ms.append((perf_counter() - start) * 1_000 / number)

    return median(samples_ms)


def print_benchmark(name, tiny_ms, torch_ms):
    ratio = tiny_ms / torch_ms
    print(f"{name:<14} TinyTorch: {tiny_ms:>9.4f} ms")
    print(f"{'':<14} PyTorch:   {torch_ms:>9.4f} ms")
    print(f"{'':<14} ratio:     {ratio:>9.2f}x (TinyTorch / PyTorch)")


mse_loss = MSELoss_CP()
mse_pred_cp, mse_target_cp = Tensor_CP(predictions), Tensor_CP(y)
mse_pred_torch, mse_target_torch = torch.from_numpy(predictions), torch.from_numpy(y)
torch_mse = nn.MSELoss()

mse_tiny_ms = benchmark(lambda: mse_loss.forward(mse_pred_cp, mse_target_cp))
mse_torch_ms = benchmark(lambda: torch_mse(mse_pred_torch, mse_target_torch))
print_benchmark("MSELoss", mse_tiny_ms, mse_torch_ms)

logits_cp, targets_cp = Tensor_CP(base_logits), Tensor_CP(class_targets)
logits_torch, targets_torch = torch.from_numpy(base_logits), torch.from_numpy(class_targets)
torch_ce = nn.CrossEntropyLoss()

ce_tiny_ms = benchmark(lambda: cross_entropy.forward(logits_cp, targets_cp))
ce_torch_ms = benchmark(lambda: torch_ce(logits_torch, targets_torch))
print_benchmark("CrossEntropy", ce_tiny_ms, ce_torch_ms)

MSELoss        TinyTorch:    0.0046 ms
               PyTorch:      0.0090 ms
               ratio:          0.51x (TinyTorch / PyTorch)


CrossEntropy   TinyTorch:    0.0393 ms
               PyTorch:      0.0352 ms
               ratio:          1.12x (TinyTorch / PyTorch)


### Results

- `MSELoss_CP` matches `torch.nn.MSELoss` on the regression data, ranks models correctly (good < baseline < bad), and is minimized at the true slope.
- `CrossEntropy` matches `torch.nn.CrossEntropyLoss` and its `log_softmax` matches `torch.log_softmax`; the log-sum-exp trick keeps the loss finite for very large logits, and the loss falls as the model grows more confident about the correct class.
- In these small CPU benchmarks `MSELoss_CP` is competitive with PyTorch because it is a single vectorized NumPy expression, while `CrossEntropy` is somewhat slower than PyTorch's fused kernel. Timings depend on CPU, thread settings, and system load, so rerun the benchmark cell on the target machine.